# 10 · Candidatos finales: un conjunto de hasta 10 envíos

**Cambio de estrategia:** la competencia deja **seleccionar hasta 10 envíos** para el leaderboard final, y el puntaje final es **el mejor de los seleccionados**. Ya no se busca *un* modelo final, sino un **conjunto** de hasta 10 envíos buenos y diversos.

**Reglas de siempre:**
- nada se entrena con el test;
- todo ensamble sigue el protocolo del 07 (umbrales elegidos por CV; `folds_seed2` solo confirma);
- la regla de comillas (D34), validada en el leaderboard público (H58), se aplica a todos los candidatos nuevos.

**Ejecución:** ≈40–60 min la primera vez:
- los modelos finales de los miembros nuevos del pool v2 del 09 (redes con 10 semillas, configuración por etiqueta, condicional y powerset);
- los modelos por fold del candidato *fold-bag*.

Todo queda en la caché de `models/10_candidatos/cache/`; `RECOMPUTE = True` la borra.

## 1. Estrategia de selección (fijada antes de calcular nada)

**Por qué conviene la diversidad.** El puntaje final es el **máximo** de los 10 seleccionados en la parte privada del test.
- **Si los 10 fueran casi idénticos,** sus puntajes privados serían casi iguales: elegir 10 equivaldría a elegir 1.
- **Si difieren en decisiones reales,** el máximo aprovecha que cada uno acierta tuits distintos, y que la parte privada favorecerá a unos más que a otros por azar.
- **Pero la diversidad sola no basta:** un candidato claramente peor casi nunca será el máximo, así que solo ocupa un hueco.
- **Por eso:** candidatos buenos (criterio a) y distintos entre sí (criterio b).

**Criterios de admisión (fijados de antemano):**
- **a) Calidad:** F1 macro **con la regla de comillas** (protocolo del 07) a ≤ 0.010 del mejor candidato, en `folds.csv` **y** en `folds_seed2`.
- **b) Diversidad:** frente a cada candidato ya admitido con mejor CV, sus predicciones en el test deben diferir en **al menos el 1% de las celdas** (1154 × 3 = 3462 celdas, es decir, ≥ 35). Si no, es un casi duplicado y se descarta.
- **c) Orden:** por la CV media de los dos conjuntos de folds.
  - El leaderboard público solo sirve para **descartar** candidatos claramente peores (más de 0.010 por debajo del mejor puntaje público), **nunca para ordenar**.
  - Con una parte del test, diferencias menores están dentro del ruido (H44).
- **d) Excepciones:**
  - FINAL_A (sin regla) solo ocupa un hueco **si sobra**, como cobertura por si la regla no se sostuviera en la parte privada;
  - `probe_HS_only` no se selecciona nunca;
  - los envíos antiguos que no cumplan a) no se seleccionan si hay suficientes candidatos que sí cumplen.
  - Se reserva **un hueco para FINAL_H_quote**, por si se autoriza HatEval (el notebook 11).

**Cómo se calcula "con la regla":** como en el 08. Las decisiones fuera de fold del crossfit (umbral de meseta elegido sin mirar el fold y jerarquía) llevan AG=1 en los tuits con comillas, y el modelo final se construye igual: combinador y umbrales del crossfit, más `quote_rule=True`.

## 0. Configuración

In [1]:
import json
import os
import platform
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import Parallel, delayed

import ensemble as en
import modeling as md
import neural as nn_
import preprocessing as pp
from common import LABELS, PROCESSED_DIR, SEED, load_test, load_train, make_folds

RECOMPUTE = False  # True: borra la caché del 10 y recalcula todo (≈40–60 min). False: reutiliza lo ya calculado
RECOMPUTE = RECOMPUTE or os.environ.get("TWEETTOXIC_RECOMPUTE") == "1"
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 40)
N_JOBS, FOLD_THREADS, FINAL_THREADS = 5, 2, 8
OUT_DIR = md.model_folder("10")
CACHE_DIR = OUT_DIR / "cache"
if RECOMPUTE and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_07, CACHE_09 = md.MODELS_DIR / "07_ensamble" / "cache", md.MODELS_DIR / "09_mejoras" / "cache"
P_COLS = [f"p_{lab}" for lab in LABELS]
AG = LABELS.index("AG")

train, test = load_train(), load_test()
Y = train[LABELS].to_numpy()
texts = train["text"]
folds = make_folds(train).to_numpy()
folds2 = make_folds(train, seed=SEED + 1, path=PROCESSED_DIR / "folds_seed2.csv").to_numpy()
FOLDSETS = {"folds": folds, "folds_seed2": folds2}
pp.load_lemma_cache(PROCESSED_DIR / "lemma_cache.parquet")
EMPTY_TR = md.empty_mask(texts)
QUOTED, QUOTED_TE = md.quoted_mask(texts), md.quoted_mask(test["text"])
REG = pd.read_csv(md.REGISTRY_PATH).set_index("model_id")
GRIDS_04 = {  # rejillas de la CV anidada del 04 (sin cambios)
    "logreg": [{"C": c} for c in (0.3, 1.0, 3.0, 10.0, 30.0)] + [{"C": c, "l1_ratio": 0.5} for c in (1.0, 3.0, 10.0)]
              + [{"C": c, "l1_ratio": 1.0} for c in (3.0, 10.0)],
    "linsvm": [{"C": c} for c in (0.03, 0.1, 0.3, 1.0, 3.0)],
}

CORE = ["V2__logreg__nested", "V1__logreg__nested", "V3__linsvm__nested"]
CLASSIC = ["V3__et__word__k2000", "V3__xgb__word__k500", "V3__nbsvm", "V3__mnb__word"]
NETS = ["V1__nn_cnn_char__tune-d0.3-lr0.002", "V1__nn_cnn_word__w2v", "V3__nn_mlp__tune-d0.5-lr0.002"]
POOL_D = CORE + CLASSIC + NETS                      # el pool del 07 (FINAL_A)
POOL_C = CORE + NETS                                # núcleo lineal + redes (FINAL_C)
S10 = {NETS[0]: NETS[0] + "__s10", NETS[1]: NETS[1] + "__s10"}
POOL_DS10 = [S10.get(m, m) for m in POOL_D]         # el pool del 07 con las CNN de 10 semillas
NEW_09 = ["V2__labelwise", "V2__conditional", "V2__powerset", "V3__nn_cnn_char__tune-d0.3-lr0.002__s10"]
POOL_V2 = POOL_DS10 + NEW_09                        # el pool v2 del 09 (14 modelos)

oof_store, test_store = pd.read_parquet(md.OOF_PATH), pd.read_parquet(md.TEST_PRED_PATH)


def stored(store, mid, ids):
    return store[store["model_id"] == mid].set_index("id").loc[np.asarray(ids), P_COLS].to_numpy()


PD = {"folds": {m: stored(oof_store, m, train["id"]) for m in POOL_D},
      "folds_seed2": {m: np.load(CACHE_07 / f"{m}__folds_seed2.npy") for m in POOL_D}}
for m in list(S10.values()) + NEW_09:
    for fs in FOLDSETS:
        PD[fs][m] = np.load(CACHE_09 / f"{m}__{fs}.npy")
P_TEST = {m: stored(test_store, m, test["id"]) for m in POOL_D}
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | SEED = {SEED} | RECOMPUTE = {RECOMPUTE}")
print(f"OOF disponibles: {len(PD['folds'])} modelos en folds.csv y {len(PD['folds_seed2'])} en folds_seed2")

Python 3.14.6 | scikit-learn 1.9.1 | SEED = 42 | RECOMPUTE = False
OOF disponibles: 16 modelos en folds.csv y 16 en folds_seed2


In [2]:
def in_worker(fn, *args):
    """Ejecuta fn en un proceso nuevo (D33: torch y xgboost/LightGBM no conviven en un mismo proceso)."""
    nn_._fresh_workers()
    out = Parallel(n_jobs=2)([delayed(fn)(*args)])[0]
    nn_._fresh_workers()
    return out


def crossfit_rule(ids, method, fs, rule=True):
    """Crossfit del 07 y, con rule, la regla de comillas aplicada a las decisiones fuera de fold (como en el 08).
    Devuelve el resultado del crossfit y las métricas por fold."""
    P = np.stack([PD[fs][m] for m in ids], axis=1)
    r = en.crossfit(en.make_combiner(method, "platt"), P, Y, FOLDSETS[fs], EMPTY_TR)
    pred = r["pred"].copy()
    if rule:
        pred[QUOTED, AG] = 1
    f = FOLDSETS[fs]
    per_fold = pd.DataFrame([en.metrics(Y[f == k], pred[f == k]) for k in sorted(np.unique(f))])
    return r, per_fold

## 2. Candidatos

| Candidato | Pool | Método | Regla | Estado |
|---|---|---|---|---|
| FINAL_A_quote | D: pool del 07 (10 modelos) | stacking de las 3 etiquetas | sí | ya enviado (0.78598) |
| FINAL_A | D | stacking de las 3 etiquetas | no | ya enviado (0.78273) |
| FINAL_C | C: núcleo lineal + redes (6) | stacking de las 3 etiquetas | no | ya enviado (0.77760) |
| **FINAL_C_quote** | C | stacking de las 3 etiquetas | sí | nuevo |
| **A2_quote** | pool v2 del 09 (14 modelos) | stacking de las 3 etiquetas | sí | nuevo |
| **D_s10_quote** | D con las 2 CNN de 10 semillas | stacking de las 3 etiquetas | sí | nuevo |
| **D_ownlabel_quote** | D | stacking con la etiqueta propia (vi) | sí | nuevo |
| **D_weights_quote** | D | pesos ≥ 0 por log-loss (iii) | sí | nuevo |
| **A_foldbag_quote** | D, con *fold-bag* en el test | stacking de las 3 etiquetas | sí | nuevo |

**Sobre A2 y las 10 semillas.** El pool v2 del 09 **ya usa** las CNN de 10 semillas, así que "pool v2 con las CNN de 10 semillas" sería el mismo candidato que A2_quote. En su lugar, **D_s10_quote** aísla el efecto de las semillas: es el pool del 07 con solo las 2 CNN cambiadas por sus versiones de 10 semillas (en el 09: +0.0014 en el stacking, H63).

**Fold-bag (A_foldbag_quote):**
- **Qué cambia:** el stacker de FINAL_A y sus umbrales se aprendieron con las **OOF**, que son predicciones de modelos entrenados con el 80% de los datos (4 folds). En el test, en cambio, cada modelo base es uno **reentrenado con el 100%**, que da probabilidades algo más extremas (ha visto más datos). Esa diferencia de distribución puede desplazar las decisiones respecto de lo que se calibró.
- **El fold-bag la elimina:** la probabilidad de test de cada modelo base es la **media de sus 5 modelos de fold**, los mismos que produjeron las OOF. El stacker y los umbrales son los de FINAL_A.
- **Su CV es por construcción la de FINAL_A_quote.** Solo cambian las predicciones del test.
- **Comprobación:** los modelos de fold deben reproducir exactamente las OOF guardadas.

In [3]:
def _fit_final_member(name, kind, spec):
    """Modelo final (100% del train) de un miembro nuevo del pool v2; lo guarda en models/10_candidatos/
    y devuelve sus probabilidades en el test. Se ejecuta en un proceso aparte."""
    import labelwise as lw_
    import modeling as md_

    if kind == "net":
        import neural as nn__  # solo aquí (D33)

        base, changes, epochs = spec
        params = nn__.load_neural(md_.find_model_path(base)).get_params()
        params.update(changes, fixed_epochs=epochs, thresholds=0.5, n_threads=FINAL_THREADS)
        model = nn__.NeuralTweetClassifier(**params).fit(texts, Y)
        nn__.save_neural_model(model, name, {"notebook": "10", "origin": "09 (pool v2)"}, notebook="10")
    elif kind == "labelwise":
        configs, _ = lw_.select_label_configs(texts, Y, n_jobs=3)
        model = lw_.LabelwiseClassifier(configs).fit(texts, Y)
        lw_.save_labelwise(model, name, {"notebook": "10", "configs": configs}, notebook="10")
    else:
        model = {"V2__conditional": lw_.ConditionalClassifier, "V2__powerset": lw_.PowersetClassifier}[name]().fit(texts, Y)
        lw_.save_labelwise(model, name, {"notebook": "10"}, notebook="10")
    return model.predict_proba(test["text"])


MEMBERS = {
    S10[NETS[0]]: ("net", (NETS[0], {"n_seeds": 10}, None)),
    S10[NETS[1]]: ("net", (NETS[1], {"n_seeds": 10}, None)),
    "V3__nn_cnn_char__tune-d0.3-lr0.002__s10": ("net", (NETS[0], {"n_seeds": 10, "variant": "V3"}, None)),
    "V2__labelwise": ("labelwise", None), "V2__conditional": ("sk", None), "V2__powerset": ("sk", None),
}
for name, (kind, spec) in sorted(MEMBERS.items(), key=lambda kv: kv[1][0] == "net"):  # scikit-learn primero
    path = CACHE_DIR / f"{name}__test.npy"
    if path.exists() and md.find_model_path(name) is not None:
        P_TEST[name] = np.load(path)
        continue
    if kind == "net":
        base, changes, _ = spec
        spec = (base, changes, json.loads((CACHE_09 / f"{name}__folds__epochs.json").read_text()))
    start = time.perf_counter()
    print(f"  final {name} ...", end=" ", flush=True)
    P_TEST[name] = in_worker(_fit_final_member, name, kind, spec)
    np.save(path, P_TEST[name])
    print(f"{time.perf_counter() - start:.0f} s", flush=True)
print("Probabilidades de test disponibles:", len(P_TEST), "modelos")

Probabilidades de test disponibles: 16 modelos


In [4]:
def _fold_fit(mid, kind, k, trn, val, path):
    """Entrena el modelo de fold k de un miembro del pool, lo guarda y devuelve (val, p_val, p_test)."""
    import joblib
    import modeling as md_

    if kind == "net":
        import neural as nn__

        params = nn__.load_neural(md_.find_model_path(mid)).get_params()
        params.update(fixed_epochs=None, thresholds=0.5, n_threads=FOLD_THREADS)
        model = nn__.NeuralTweetClassifier(**params)
    elif kind == "nested":
        r = REG.loc[mid]
        best, _ = md_.select_params(r["variant"], r["classifier"], r["features"], GRIDS_04[r["classifier"]],
                                    texts[trn].reset_index(drop=True), Y[trn])
        model = md_.TweetToxicClassifier(variant=r["variant"], classifier=r["classifier"], features=r["features"], clf_params=best)
    else:
        r = REG.loc[mid]
        model = md_.TweetToxicClassifier(variant=r["variant"], classifier=r["classifier"], features=r["features"],
                                         clf_params=json.loads(r["params"]) or None)
    model.fit(texts[trn], Y[trn])
    joblib.dump(model, path, compress=3)
    return val, model.predict_proba(texts[val]), model.predict_proba(test["text"])


def _save_foldbag(mid, paths, uses_torch):
    import joblib
    import ensemble as en_
    import modeling as md_

    name = f"{mid}__foldbag"
    bag = en_.FoldBagModel([joblib.load(p) for p in paths], name=name)
    path = md_.model_folder("10") / f"{name}.joblib"
    joblib.dump(bag, path, compress=3)
    path.with_suffix(".json").write_text(json.dumps({"name": name, "base": mid, "folds": len(paths), "uses_torch": uses_torch,
                                                     "notebook": "10", "role": "fold-bag (A_foldbag_quote)"}, indent=2))
    for p in paths:
        p.unlink()
    return name


def kind_of(mid):
    return "net" if mid in NETS else "nested" if mid.endswith("__nested") else "sk"


FOLDBAG_TEST, rows = {}, []
ks = sorted(np.unique(folds))
for mid in sorted(POOL_D, key=lambda m: kind_of(m) == "net"):  # scikit-learn primero
    test_path, val_path = CACHE_DIR / f"{mid}__foldbag__test.npy", CACHE_DIR / f"{mid}__foldbag__val.npy"
    if not (test_path.exists() and md.find_model_path(f"{mid}__foldbag") is not None):
        start = time.perf_counter()
        print(f"  fold-bag {mid} ...", end=" ", flush=True)
        paths = [CACHE_DIR / f"{mid}__fold{k}.joblib" for k in ks]
        nn_._fresh_workers()
        res = Parallel(n_jobs=N_JOBS)(delayed(_fold_fit)(mid, kind_of(mid), k, np.where(folds != k)[0], np.where(folds == k)[0], p)
                                      for k, p in zip(ks, paths))
        nn_._fresh_workers()
        val_pred = np.zeros(Y.shape)
        for val, p_val, _ in res:
            val_pred[val] = p_val
        np.save(val_path, val_pred)
        np.save(test_path, np.mean([p_te for _, _, p_te in res], axis=0))
        in_worker(_save_foldbag, mid, paths, kind_of(mid) == "net")
        print(f"{time.perf_counter() - start:.0f} s", flush=True)
    FOLDBAG_TEST[mid] = np.load(test_path)
    rows.append({"modelo": mid, "máx. |OOF de los modelos de fold − OOF guardada|": float(np.abs(np.load(val_path) - PD["folds"][mid]).max()),
                 "media |p test fold-bag − p test 100%|": round(float(np.abs(FOLDBAG_TEST[mid] - P_TEST[mid]).mean()), 4)})
fb = pd.DataFrame(rows).set_index("modelo")
display(fb)
print("Los modelos de fold reproducen las OOF (≤ 1e-6; las redes difieren en ≈1e-7 por redondeo en float32):",
      bool((fb.iloc[:, 0] <= 1e-6).all()))

,máx. |OOF de los modelos de fold − OOF guardada|,media |p test fold-bag − p test 100%|
modelo,,
V2__logreg__nested,0.000000e+00,0.0184
V1__logreg__nested,0.000000e+00,0.0604
V3__linsvm__nested,0.000000e+00,0.0179
V3__et__word__k2000,0.000000e+00,0.0334
V3__xgb__word__k500,0.000000e+00,0.0280
V3__nbsvm,0.000000e+00,0.0145
V3__mnb__word,0.000000e+00,0.0094
V1__nn_cnn_char__tune-d0.3-lr0.002,7.947286e-08,0.0422
V1__nn_cnn_word__w2v,7.947286e-08,0.0393


Los modelos de fold reproducen las OOF (≤ 1e-6; las redes difieren en ≈1e-7 por redondeo en float32): True


In [5]:
CANDS = {  # etiqueta: (pool, método, con regla, id del modelo ya existente o None)
    "FINAL_A_quote": (POOL_D, "stack-all", True, "ens08__D__stack-all__quote"),
    "FINAL_A": (POOL_D, "stack-all", False, "ens07__D__stack-all"),
    "FINAL_C": (POOL_C, "stack-all", False, "ens08__C__stack-all"),
    "FINAL_C_quote": (POOL_C, "stack-all", True, None),
    "A2_quote": (POOL_V2, "stack-all", True, None),
    "D_s10_quote": (POOL_DS10, "stack-all", True, None),
    "D_ownlabel_quote": (POOL_D, "stack-own", True, None),
    "D_weights_quote": (POOL_D, "weights", True, None),
    "A_foldbag_quote": (POOL_D, "stack-all", True, None),
}
NAMES = {"FINAL_C_quote": "ens10__C__stack-all__quote", "A2_quote": "ens10__v2__stack-all__quote",
         "D_s10_quote": "ens10__Ds10__stack-all__quote", "D_ownlabel_quote": "ens10__D__stack-own__quote",
         "D_weights_quote": "ens10__D__weights__quote", "A_foldbag_quote": "ens10__D__stack-all__foldbag__quote"}
CV, FILES, MODELS_10, rows = {}, {}, {}, []
for tag, (ids, method, rule, existing) in CANDS.items():
    res = {fs: crossfit_rule(ids, method, fs, rule) for fs in FOLDSETS}
    CV[tag] = {fs: pf for fs, (r, pf) in res.items()}
    if existing:
        FILES[tag] = {"FINAL_A_quote": "FINAL_A_quote_ens08__D__stack-all__quote.csv", "FINAL_A": "FINAL_A_ens07__D__stack-all.csv",
                      "FINAL_C": "FINAL_C_ens08__C__stack-all.csv"}[tag]
        continue
    name = NAMES[tag]
    P_main = np.stack([PD["folds"][m] for m in ids], axis=1)
    if tag == "A_foldbag_quote":  # mismo stacker y umbrales que FINAL_A; modelos base fold-bag
        fa = md.load_model("ens07__D__stack-all")
        base_ids, fitted, thr = [f"{m}__foldbag" for m in ids], fa.combiner, fa.thresholds
        P_te = np.stack([FOLDBAG_TEST[m] for m in ids], axis=1)
    else:
        fitted, thr = en.fit_final(en.make_combiner(method, "platt"), P_main, Y, folds, res["folds"][0]["proba"])
        base_ids = ids
        P_te = np.stack([P_TEST[m] for m in ids], axis=1)
    model = en.EnsembleClassifier(base_ids, fitted, thr, name=name, quote_rule=True)
    cv_info = {fs: round(pf["F1_macro"].mean(), 4) for fs, pf in CV[tag].items()}
    en.save_ensemble(model, name, {"notebook": "10", "role": tag, "pool": ids, "method": method, "cv_F1_macro_with_rule": cv_info,
                                   "seed": SEED})
    sub_path = md.make_submission(name, filename=f"{tag}_{name}")
    sub = pd.read_csv(sub_path)
    expected = md.decide(model.predict_proba_from_base(P_te), test["text"], thr)
    expected[QUOTED_TE, AG] = 1
    p_disk = md.load_model(name).predict_proba_from_base(P_te)
    rows.append({"candidato": tag, "archivo": sub_path.name, "filas": len(sub), "ids en orden": bool((sub["id"] == test["id"]).all()),
                 "valores": sorted(np.unique(sub[LABELS]).tolist()), "nulos": int(sub.isna().sum().sum()),
                 "= probabilidades guardadas": bool((sub[LABELS].to_numpy() == expected).all()),
                 "recarga |disco − memoria|": float(np.abs(p_disk - model.predict_proba_from_base(P_te)).max()),
                 "umbrales": thr})
    FILES[tag] = sub_path.name
    MODELS_10[tag] = {"name": name, "model": model, "P_te": P_te, "oof": res["folds"][0]["proba"], "ids": ids}
checks = pd.DataFrame(rows).set_index("candidato")
display(checks)
assert checks["= probabilidades guardadas"].all() and (checks["recarga |disco − memoria|"] == 0).all()

,archivo,filas,ids en orden,valores,nulos,= probabilidades guardadas,recarga |disco − memoria|,umbrales
candidato,,,,,,,,
FINAL_C_quote,FINAL_C_quote_ens10__C__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.39, 'TR': 0.36, 'AG': 0.37}"
A2_quote,A2_quote_ens10__v2__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.35, 'TR': 0.37, 'AG': 0.39}"
D_s10_quote,D_s10_quote_ens10__Ds10__stack-all__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.37, 'TR': 0.34, 'AG': 0.37}"
D_ownlabel_quote,D_ownlabel_quote_ens10__D__stack-own__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.39, 'TR': 0.37, 'AG': 0.36}"
D_weights_quote,D_weights_quote_ens10__D__weights__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.36, 'TR': 0.35, 'AG': 0.35}"
A_foldbag_quote,A_foldbag_quote_ens10__D__stack-all__foldbag__quote.csv,1154,True,"[0, 1]",0,True,0.0,"{'HS': 0.37, 'TR': 0.35, 'AG': 0.36}"


**Interpretación:**
- **Modelos finales de los miembros nuevos del pool v2:** se entrenan una vez con todo el train y quedan en la caché, así que las ejecuciones siguientes no los repiten.
- **Fold-bag:**
  - los modelos de fold reproducen las OOF guardadas (diferencia 0 en los de scikit-learn, ≈8·10⁻⁸ en las redes, que es redondeo en float32);
  - en el test, sus probabilidades se separan de las del modelo reentrenado con el 100% en 0.009–0.060 de media (la mayor, la logística V1 con 0.060; las redes, 0.04–0.056). Es justo la diferencia de distribución que el fold-bag quiere evitar.
- **Comprobaciones de los 6 candidatos nuevos:**
  - el archivo coincide con las probabilidades guardadas;
  - la recarga desde disco da diferencia 0;
  - 1154 filas, ids en orden, valores 0/1 y sin nulos.

## 3. Tabla de candidatos y selección

Se aplican los criterios de la sección 1, en el orden fijado, a los candidatos nuevos, a los ya enviados y a los envíos antiguos de un solo modelo.

- **Diferencia en el test:** % de las 3462 celdas (tuit × etiqueta) en que dos archivos no coinciden.
- **Puntajes públicos conocidos** (`PROGRESO.md` §8): solo se usan para el criterio de descarte c).

In [6]:
LB = {"FINAL_A_quote": 0.78598, "FINAL_A": 0.78273, "FINAL_C": 0.77760, "FINAL_B": 0.76466,
      "V2__logreg__nested (04)": 0.77714, "V3__linsvm (03)": 0.77464}
OLD_FILES = {"FINAL_B": "FINAL_B_single07__V1__logreg__nested.csv", "V2__logreg__nested (04)": "V2__logreg__nested.csv",
             "V3__linsvm (03)": "V3__linsvm.csv"}
OLD_CV = {"FINAL_B": (["V1__logreg__nested"], False), "V2__logreg__nested (04)": (["V2__logreg__nested"], False),
          "V3__linsvm (03)": (["V3__linsvm"], False)}
oof_old = {m: stored(oof_store, m, train["id"]) for m in ["V1__logreg__nested", "V3__linsvm"]}
PD["folds"].update(oof_old)
PD["folds_seed2"]["V1__logreg__nested"] = np.load(CACHE_07 / "V1__logreg__nested__folds_seed2.npy")
for tag, (ids, rule) in OLD_CV.items():
    CV[tag] = {fs: crossfit_rule(ids, "mean", fs, rule)[1] if all(m in PD[fs] for m in ids) else None for fs in FOLDSETS}
FILES.update(OLD_FILES)
SUBS = {tag: pd.read_csv(md.SUBMISSIONS_DIR / f)[LABELS].to_numpy() for tag, f in FILES.items()}

rows = []
for tag in FILES:
    main, s2 = CV[tag]["folds"], CV[tag]["folds_seed2"]
    rule = CANDS[tag][2] if tag in CANDS else False
    rows.append({"candidato": tag, "regla": "sí" if rule else "no",
                 "CV folds": round(main["F1_macro"].mean(), 4), "CV folds_seed2": round(s2["F1_macro"].mean(), 4) if s2 is not None else np.nan,
                 "LB público": LB.get(tag, np.nan),
                 "% celdas ≠ FINAL_A_quote": round(100 * (SUBS[tag] != SUBS["FINAL_A_quote"]).mean(), 2)})
TABLE = pd.DataFrame(rows).set_index("candidato")
TABLE["CV media"] = TABLE[["CV folds", "CV folds_seed2"]].mean(axis=1, skipna=False)
TABLE = TABLE.sort_values("CV media", ascending=False, na_position="last")
order = list(TABLE.index)
DIFF = pd.DataFrame([[round(100 * (SUBS[a] != SUBS[b]).mean(), 2) for b in order] for a in order], index=order, columns=order)

In [7]:
SLOTS, RESERVED = 10, 1  # un hueco reservado para FINAL_H_quote (HatEval, si se autoriza)
best_main = TABLE.loc[TABLE["regla"] == "sí", "CV folds"].max()
best_s2 = TABLE.loc[TABLE["regla"] == "sí", "CV folds_seed2"].max()
best_lb = max(LB.values())
admitted, status = [], {}
for tag in order:
    row = TABLE.loc[tag]
    a = (best_main - row["CV folds"] <= 0.010) and (not np.isnan(row["CV folds_seed2"]) and best_s2 - row["CV folds_seed2"] <= 0.010)
    c = np.isnan(row["LB público"]) or row["LB público"] >= best_lb - 0.010
    near = [adm for adm in admitted if DIFF.loc[tag, adm] < 1.0]
    if tag == "FINAL_A":
        status[tag] = "cobertura de la regla (d): solo si sobra hueco"
        continue
    if not a:
        status[tag] = "no cumple a) (calidad)"
    elif not c:
        status[tag] = "descartado por c) (LB público > 0.010 bajo el mejor)"
    elif near:
        status[tag] = f"casi duplicado de {near[0]} (b)"
    else:
        admitted.append(tag)
        status[tag] = "admitido"
TABLE["a) calidad"] = [(best_main - TABLE.loc[t, "CV folds"] <= 0.010) and (best_s2 - TABLE.loc[t, "CV folds_seed2"] <= 0.010)
                       if not np.isnan(TABLE.loc[t, "CV folds_seed2"]) else False for t in order]
TABLE["b) ≥1% distinto de los admitidos mejores"] = [not status[t].startswith("casi") for t in order]
TABLE["c) LB no descartado"] = [np.isnan(TABLE.loc[t, "LB público"]) or TABLE.loc[t, "LB público"] >= best_lb - 0.010 for t in order]
TABLE["estado"] = [status[t] for t in order]
print(f"Mejor CV con la regla: folds {best_main:.4f} | folds_seed2 {best_s2:.4f} | mejor LB público {best_lb:.5f}")
display(TABLE)
print("% de celdas distintas entre los archivos de test (3462 celdas):")
display(DIFF)

Mejor CV con la regla: folds 0.8072 | folds_seed2 0.8052 | mejor LB público 0.78598


,regla,CV folds,CV folds_seed2,LB público,% celdas ≠ FINAL_A_quote,CV media,a) calidad,b) ≥1% distinto de los admitidos mejores,c) LB no descartado,estado
candidato,,,,,,,,,,
A2_quote,sí,0.8072,0.8052,NaN,3.18,0.80620,True,True,True,admitido
D_s10_quote,sí,0.8069,0.8014,NaN,2.25,0.80415,True,True,True,admitido
FINAL_A_quote,sí,0.8060,0.8019,0.78598,0.00,0.80395,True,True,True,admitido
A_foldbag_quote,sí,0.8060,0.8019,NaN,2.34,0.80395,True,True,True,admitido
D_ownlabel_quote,sí,0.8053,0.8008,NaN,3.52,0.80305,True,True,True,admitido
FINAL_C_quote,sí,0.8027,0.8017,NaN,3.03,0.80220,True,True,True,admitido
D_weights_quote,sí,0.8041,0.7995,NaN,4.10,0.80180,True,True,True,admitido
FINAL_A,no,0.8008,0.7961,0.78273,0.14,0.79845,True,True,True,cobertura de la regla (d): solo si sobra hueco
FINAL_C,no,0.7965,0.7981,0.77760,3.15,0.79730,False,True,True,no cumple a) (calidad)


% de celdas distintas entre los archivos de test (3462 celdas):


,A2_quote,D_s10_quote,FINAL_A_quote,A_foldbag_quote,D_ownlabel_quote,FINAL_C_quote,D_weights_quote,FINAL_A,FINAL_C,FINAL_B,V2__logreg__nested (04),V3__linsvm (03)
A2_quote,0.00,1.96,3.18,3.32,3.64,3.73,4.27,3.32,3.84,6.79,6.85,7.80
D_s10_quote,1.96,0.00,2.25,3.03,3.81,3.38,4.16,2.40,3.50,6.85,6.67,8.15
FINAL_A_quote,3.18,2.25,0.00,2.34,3.52,3.03,4.10,0.14,3.15,6.38,6.67,7.74
A_foldbag_quote,3.32,3.03,2.34,0.00,2.92,3.64,3.32,2.48,3.76,5.20,5.95,6.56
D_ownlabel_quote,3.64,3.81,3.52,2.92,0.00,3.38,2.89,3.67,3.50,4.82,4.94,5.89
FINAL_C_quote,3.73,3.38,3.03,3.64,3.38,0.00,3.84,3.18,0.12,5.60,6.53,7.42
D_weights_quote,4.27,4.16,4.10,3.32,2.89,3.84,0.00,4.25,3.96,4.19,6.21,7.45
FINAL_A,3.32,2.40,0.14,2.48,3.67,3.18,4.25,0.00,3.06,6.30,6.59,7.65
FINAL_C,3.84,3.50,3.15,3.76,3.50,0.12,3.96,3.06,0.00,5.49,6.41,7.31
FINAL_B,6.79,6.85,6.38,5.20,4.82,5.60,4.19,6.30,5.49,0.00,4.79,7.48


**Interpretación:**
- **Los 7 candidatos con la regla cumplen los tres criterios.**
  - CV media de 0.8018 a 0.8062.
  - El mejor es A2_quote: 0.8072 en `folds.csv` y 0.8052 en `folds_seed2`.
  - El más débil, D_weights_quote (0.8041 / 0.7995), sigue a menos de 0.006 del mejor.
- **Todos son diversos entre sí:** se diferencian en un 1.96–4.27% de las celdas (68–148 de 3462). El par más parecido es A2_quote / D_s10_quote (1.96%), lógico porque comparten las CNN de 10 semillas.
- **Las diferencias de CV entre ellos (≤ 0.005) están por debajo de la desviación entre folds (≈0.018).** El orden por CV es débil. Lo que justifica seleccionar varios es precisamente que no se puede saber cuál será el mejor en la parte privada.
- **FINAL_A** difiere de FINAL_A_quote solo en el 0.14% (5 tuits): es la cobertura frente a la regla, no un candidato diverso.
- **No cumplen la calidad (a):**
  - FINAL_C sin regla, por poco: 0.0107 por debajo en `folds.csv`;
  - FINAL_B y los dos envíos antiguos de un solo modelo, que además están más de 0.010 por debajo del mejor puntaje público (criterio c), salvo `V2__logreg__nested`.

In [8]:
selected = admitted[:SLOTS - RESERVED]
if len(selected) < SLOTS - RESERVED:
    selected.append("FINAL_A")  # cobertura frente a la regla (d), solo si sobra hueco
to_upload = [t for t in selected if np.isnan(TABLE.loc[t, "LB público"])]
plan = []
for i, tag in enumerate(to_upload):
    plan.append({"día": f"día {i // 3 + 1}" + (" (hoy)" if i < 3 else ""), "orden": i + 1, "candidato": tag, "archivo": FILES[tag],
                 "CV media": TABLE.loc[tag, "CV media"]})
display(pd.DataFrame(plan).set_index(["día", "orden"]) if plan else "Nada nuevo que subir")
final_list = selected + ["FINAL_H_quote (reservado: HatEval, si se autoriza)"]
free = SLOTS - len(final_list)
print(f"Selección final propuesta: {len(final_list)} de {SLOTS} ({free} hueco{'s' if free != 1 else ''} libre{'s' if free != 1 else ''})")
for i, t in enumerate(final_list, 1):
    print(f"  {i}. {t}" + (f"  [{FILES[t]}]" if t in FILES else ""))

candidato                                                  archivo  CV media
día         orden                                                                                     
día 1 (hoy) 1              A2_quote                 A2_quote_ens10__v2__stack-all__quote.csv   0.80620
            2           D_s10_quote            D_s10_quote_ens10__Ds10__stack-all__quote.csv   0.80415
            3       A_foldbag_quote  A_foldbag_quote_ens10__D__stack-all__foldbag__quote.csv   0.80395
día 2       4      D_ownlabel_quote          D_ownlabel_quote_ens10__D__stack-own__quote.csv   0.80305
            5         FINAL_C_quote             FINAL_C_quote_ens10__C__stack-all__quote.csv   0.80220
            6       D_weights_quote             D_weights_quote_ens10__D__weights__quote.csv   0.80180

Selección final propuesta: 9 de 10 (1 hueco libre)
  1. A2_quote  [A2_quote_ens10__v2__stack-all__quote.csv]
  2. D_s10_quote  [D_s10_quote_ens10__Ds10__stack-all__quote.csv]
  3. FINAL_A_quote  [FINAL_A_quote_ens08__D__stack-all__quote.csv]
  4. A_foldbag_quote  [A_foldbag_quote_ens10__D__stack-all__foldbag__quote.csv]
  5. D_ownlabel_quote  [D_ownlabel_quote_ens10__D__stack-own__quote.csv]
  6. FINAL_C_quote  [FINAL_C_quote_ens10__C__stack-all__quote.csv]
  7. D_weights_quote  [D_weights_quote_ens10__D__weights__quote.csv]
  8. FINAL_A  [FINAL_A_ens07__D__stack-all.csv]
  9. FINAL_H_quote (reservado: HatEval, si se autoriza)


In [9]:
# Registro: los modelos base nuevos (finales del pool v2) y los candidatos nuevos
reg_rows, oof_frames, test_frames = [], [], []
for name in MEMBERS:
    pf = crossfit_rule([name], "mean", "folds", rule=False)[1]
    base_id = name.replace("__s10", "")
    found = md.find_model_path(name)
    reg_rows.append({"model_id": name, "family": REG.loc[base_id, "family"] if base_id in REG.index else "lineal",
                     "classifier": name.split("__")[1], "variant": name.split("__")[0], "features": "09 (pool v2)",
                     "params": "{}", "notebook": "10", "F1_macro": round(pf["F1_macro"].mean(), 4),
                     "F1_macro_std": round(pf["F1_macro"].std(), 4), "F1_micro": round(pf["F1_micro"].mean(), 4),
                     **{f"F1_{lab}": round(pf[f"F1_{lab}"].mean(), 4) for lab in LABELS}, "proba_calibrated": False,
                     "saved": found is not None, "size_mb": round(found.stat().st_size / 1e6, 2) if found else None})
    oof_frames.append(md.prediction_frame(name, train["id"], PD["folds"][name], folds))
    test_frames.append(md.prediction_frame(name, test["id"], P_TEST[name]))
for tag, M in MODELS_10.items():
    pf = CV[tag]["folds"]
    proba_te = M["model"].predict_proba_from_base(M["P_te"])
    reg_rows.append({"model_id": M["name"], "family": "ensamble", "classifier": CANDS[tag][1], "variant": "—",
                     "features": f"{tag}: " + " + ".join(M["ids"]), "params": "{}", "notebook": "10",
                     "F1_macro": round(pf["F1_macro"].mean(), 4), "F1_macro_std": round(pf["F1_macro"].std(), 4),
                     "F1_micro": round(pf["F1_micro"].mean(), 4), **{f"F1_{lab}": round(pf[f"F1_{lab}"].mean(), 4) for lab in LABELS},
                     **{f"thr_{lab}": M["model"].thresholds[lab] for lab in LABELS}, "proba_calibrated": True, "saved": True})
    oof_frames.append(md.prediction_frame(M["name"], train["id"], M["oof"], folds, calibrated=True))
    test_frames.append(md.prediction_frame(M["name"], test["id"], proba_te, pred=SUBS[tag], calibrated=True))
reg = md.update_registry(reg_rows)
md.update_predictions(oof_frames, md.OOF_PATH)
md.update_predictions(test_frames, md.TEST_PRED_PATH)
print(f"Registro: {len(reg)} modelos | añadidos: {len(reg_rows)} (F1 de los candidatos: con la regla de comillas)")

Registro: 199 modelos | añadidos: 12 (F1 de los candidatos: con la regla de comillas)


**Plan de envíos** (3 por día; los ya subidos no se repiten):
- **Día 1 (hoy):** A2_quote, D_s10_quote y A_foldbag_quote.
- **Día 2:** D_ownlabel_quote, FINAL_C_quote y D_weights_quote.

**Selección final** (9 de 10, con 1 hueco libre):
- los 7 candidatos con la regla, incluido FINAL_A_quote (ya subido);
- FINAL_A, como cobertura de la regla;
- FINAL_H_quote, reservado para HatEval si se autoriza.

**Después de subir:**
- los puntajes públicos solo pueden **descartar** un candidato si queda más de 0.010 por debajo del mejor puntaje público (criterio c);
- **no reordenan** la lista ni deciden entre candidatos cercanos;
- el hueco libre queda para un segundo envío de HatEval o para lo que el equipo decida, siempre con estos criterios.

## 4. Conclusiones

1. **La estrategia "mejor de 10" se resuelve con diversidad entre candidatos buenos.** Los 7 candidatos con la regla están todos a ≤ 0.006 del mejor en CV y difieren entre sí en un 2–4% de las celdas del test.
2. **Candidatos nuevos:** FINAL_C_quote, A2_quote (el mejor en CV: 0.8072 / 0.8052), D_s10_quote, D_ownlabel_quote, D_weights_quote y A_foldbag_quote. Están guardados en `models/10_candidatos/`, con sus envíos en `submissions/`, y pasan todas las comprobaciones.
3. **El fold-bag** tiene la misma CV que FINAL_A_quote, pero cambia el 2.34% de las celdas del test. Es una apuesta directa a que la diferencia de distribución entre modelos de fold y modelos reentrenados importa.
4. **Plan:** 3 envíos hoy (A2_quote, D_s10_quote, A_foldbag_quote) y 3 mañana (D_ownlabel_quote, FINAL_C_quote, D_weights_quote). Selección final de 9 + 1 libre, con un hueco reservado para HatEval.